# Inference Demo

This notebook demonstrates offline inference for Cortia's post-award anomaly detector.
It loads the persisted artifacts from training, scores three curated demo cases,
and prints a human-readable explanation for each result.


In [3]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display


def print_section(title):
    print("=" * 50)
    print(title)
    print("=" * 50)


def format_number(value):
    if pd.isna(value):
        return "missing"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        return f"{value:,.4f}".rstrip("0").rstrip(".")
    return str(value)


def assign_severity(scores, medium_cutoff, anomaly_threshold):
    return np.select(
        [
            scores >= anomaly_threshold,
            scores >= medium_cutoff,
        ],
        ["high", "medium"],
        default="low",
    )


def engineer_features(frame):
    data = frame.copy()
    data["date"] = pd.to_datetime(data["date"])
    data["award_date"] = pd.to_datetime(data["award_date"])
    data["award_month"] = data["award_date"].dt.month
    data["award_quarter"] = data["award_date"].dt.quarter
    data["award_weekday"] = data["award_date"].dt.weekday
    data["log_tender_minvalue"] = np.log1p(data["tender_minvalue"])
    data["log_award_value"] = np.log1p(data["award_value"])
    data["value_gap"] = data["award_value"] - data["tender_minvalue"]
    data["title_word_count"] = data["tender_title"].fillna("").str.split().str.len()
    data["award_title_word_count"] = data["award_title"].fillna("").str.split().str.len()
    data["supplier_count"] = data["award_supplier"].fillna("").astype(str).str.split(",").str.len()
    data["award_value_per_day"] = data["award_value"] / data["days_to_award"].replace(0, 1)
    data["same_day_award_flag"] = (data["days_to_award"] == 0).astype(int)
    return data


In [ ]:
artifact_dir = Path("artifacts/post_award_anomaly")

# ngeload by folder & filenya(joblib dari training.ipynb)
model = joblib.load(artifact_dir / "isolation_forest.joblib")
preprocessor = joblib.load(artifact_dir / "preprocessor.joblib")
# treshold limit
model_config = json.loads((artifact_dir / "model_config.json").read_text(encoding="utf-8"))
# sumber reasoning ( nanti diolah ke func di bawah )
explanation_baselines = json.loads((artifact_dir / "explanation_baselines.json").read_text(encoding="utf-8"))
demo_input = pd.read_csv(artifact_dir / "demo_input_cases.csv")

print_section("LOADED ARTIFACTS")
print(f"Artifacts dir        : {artifact_dir.resolve()}")
print(f"Numeric features     : {model_config['numeric_features']}")
print(f"Categorical features : {model_config['categorical_features']}")
print(f"Medium cutoff        : {model_config['medium_cutoff']:.6f}")
print(f"Anomaly threshold    : {model_config['anomaly_threshold']:.6f}")

print("\nDemo inputs:")
display(demo_input[["demo_case_label", "award_date", "mainprocurementcategory", "award_value", "days_to_award", "budget_utilization_ratio"]])


LOADED ARTIFACTS
Artifacts dir        : C:\Users\VICTUS\coding\collab\1\cortia\artifacts\post_award_anomaly
Numeric features     : ['tender_minvalue', 'award_value', 'days_to_award', 'budget_utilization_ratio', 'value_gap', 'log_tender_minvalue', 'log_award_value', 'title_word_count', 'award_title_word_count', 'supplier_count', 'award_value_per_day', 'same_day_award_flag', 'award_month', 'award_quarter', 'award_weekday']
Categorical features : ['mainprocurementcategory']
Medium cutoff        : 0.459542
Anomaly threshold    : 0.588172

Demo inputs:


,demo_case_label,award_date,mainprocurementcategory,award_value,days_to_award,budget_utilization_ratio
0,normal_demo,2023-04-06,Services,2.233098e+09,34,0.891695
1,borderline_demo,2022-08-19,Works,3.540522e+10,28,0.800000
2,anomaly_demo,2023-01-16,Works,7.871680e+11,200,0.867022


In [ ]:
numeric_reference = explanation_baselines["numeric_reference"]
categorical_reference = explanation_baselines["categorical_reference"]
explanation_numeric_features = explanation_baselines["explanation_numeric_features"]
categorical_features = explanation_baselines["categorical_features"]


def explain_prediction(row):
    is_anomaly = row["prediction_label"] == "anomaly"
    contributions = []

    for col in explanation_numeric_features:
        baseline = float(numeric_reference[col]["median"])
        spread = float(numeric_reference[col]["spread"])
        value = row[col]
        deviation = abs(float(value) - baseline) / spread

        if is_anomaly:
            impact = "positive"
            if float(value) >= baseline:
                reason = f"{col} is higher than the normal baseline ({format_number(value)} vs {format_number(baseline)})."
            else:
                reason = f"{col} is lower than the normal baseline ({format_number(value)} vs {format_number(baseline)})."
            score = deviation
        else:
            impact = "negative"
            reason = f"{col} stays close to the normal baseline ({format_number(value)} vs {format_number(baseline)})."
            score = 1 / (1 + deviation)

        contributions.append({"feature": col, "score": float(score), "impact": impact, "reason": reason})

    for col in categorical_features:
        value = str(row[col])
        frequency = float(categorical_reference[col].get(value, 0.0))

        if is_anomaly:
            impact = "positive"
            score = 1.0 - frequency
            if frequency == 0:
                reason = f"{col}='{value}' was not seen in the normal training baseline."
            else:
                reason = f"{col}='{value}' is rare in the normal training baseline ({frequency:.2%})."
        else:
            impact = "negative"
            score = frequency
            reason = f"{col}='{value}' is common in the normal training baseline ({frequency:.2%})."

        contributions.append({"feature": col, "score": float(score), "impact": impact, "reason": reason})

    top_contributions = sorted(contributions, key=lambda item: item["score"], reverse=True)[:3]
    result = {}
    summary_lines = []
    for idx, item in enumerate(top_contributions, start=1):
        result[f"top_{idx}_feature"] = item["feature"]
        result[f"top_{idx}_impact"] = item["impact"]
        result[f"top_{idx}_reason"] = item["reason"]
        summary_lines.append(f"{idx}. {item['feature']} ({item['impact']}): {item['reason']}")

    label_text = "Anomalous" if is_anomaly else "Normal"
    result["human_readable_explanation"] = f"{label_text} transaction. " + " ".join(summary_lines)
    return pd.Series(result)

# simulasi inputnya 
demo_features = engineer_features(demo_input)
feature_columns = model_config["numeric_features"] + model_config["categorical_features"]
# Column Transfer
X_demo = preprocessor.transform(demo_features[feature_columns])
# kita issolation forest yang kayak di training jd tinggal panggil
demo_scores = -model.score_samples(X_demo)

demo_features["anomaly_score"] = demo_scores
demo_features["prediction_label"] = np.where(demo_scores >= model_config["anomaly_threshold"], "anomaly", "normal")
demo_features["anomaly_flag"] = (demo_features["prediction_label"] == "anomaly").astype(int)
demo_features["severity_band"] = assign_severity(demo_scores, model_config["medium_cutoff"], model_config["anomaly_threshold"])

demo_explanations = demo_features.apply(explain_prediction, axis=1)
demo_output = pd.concat([demo_features.reset_index(drop=True), demo_explanations.reset_index(drop=True)], axis=1)
demo_output.to_csv(artifact_dir / "demo_inference_results.csv", index=False)

print_section("DEMO PREDICTIONS")
display(
    demo_output[
        [
            "demo_case_label",
            "award_date",
            "buyer_name",
            "mainprocurementcategory",
            "award_value",
            "days_to_award",
            "budget_utilization_ratio",
            "anomaly_score",
            "severity_band",
            "prediction_label",
            "human_readable_explanation",
        ]
    ]
)


DEMO PREDICTIONS


,demo_case_label,award_date,buyer_name,mainprocurementcategory,award_value,days_to_award,budget_utilization_ratio,anomaly_score,severity_band,prediction_label,human_readable_explanation
0,normal_demo,2023-04-06,Pemerintah Daerah Provinsi Dki Jakarta,Services,2.233098e+09,34,0.891695,0.432089,low,normal,Normal transaction. 1. supplier_count (negativ...
1,borderline_demo,2022-08-19,Pemerintah Daerah Provinsi Dki Jakarta,Works,3.540522e+10,28,0.800000,0.587034,medium,normal,Normal transaction. 1. supplier_count (negativ...
2,anomaly_demo,2023-01-16,Pemerintah Daerah Provinsi Dki Jakarta,Works,7.871680e+11,200,0.867022,0.767747,high,anomaly,Anomalous transaction. 1. value_gap (positive)...


In [6]:
print_section("CASE WALKTHROUGHS")
for _, row in demo_output.iterrows():
    print(f"Case        : {row['demo_case_label']}")
    print(f"Prediction  : {row['prediction_label']}")
    print(f"Severity    : {row['severity_band']}")
    print(f"Score       : {row['anomaly_score']:.6f}")
    print(f"Top Driver  : {row['top_1_feature']} ({row['top_1_impact']})")
    print(f"Explanation : {row['human_readable_explanation']}")
    print("-" * 50)

print("Saved demo output:")
print((artifact_dir / "demo_inference_results.csv").resolve())


CASE WALKTHROUGHS
Case        : normal_demo
Prediction  : normal
Severity    : low
Score       : 0.432089
Top Driver  : supplier_count (negative)
Explanation : Normal transaction. 1. supplier_count (negative): supplier_count stays close to the normal baseline (1 vs 1). 2. title_word_count (negative): title_word_count stays close to the normal baseline (8 vs 8). 3. budget_utilization_ratio (negative): budget_utilization_ratio stays close to the normal baseline (0.8917 vs 0.89).
--------------------------------------------------
Case        : borderline_demo
Prediction  : normal
Severity    : medium
Score       : 0.587034
Top Driver  : supplier_count (negative)
Explanation : Normal transaction. 1. supplier_count (negative): supplier_count stays close to the normal baseline (1 vs 1). 2. title_word_count (negative): title_word_count stays close to the normal baseline (7 vs 8). 3. days_to_award (negative): days_to_award stays close to the normal baseline (28 vs 21).
------------------------